# DeepSearch — Local & Free
Uses **Ollama** (local LLM) + **DuckDuckGo** (free search) + **BeautifulSoup** (web scraping).

**Flow:**
1. Generate targeted sub-queries for your topic
2. Search the web via DuckDuckGo
3. Scrape & chunk page content
4. Rank & filter relevant chunks
5. Synthesize a final deep report

## 1. Setup & Config

In [ ]:
# !pip install ddgs 
# !pip install requests beautifulsoup4 ollama yt-dlp -q
# !pip show psycopg2-binary 2>/dev/null | head -2; pg_isready 2>/dev/null || echo "pg_isready not found"; psql --version 2>/dev/null || echo "psql not found"

  Using cached ddgs-9.11.4-py3-none-any.whl.metadata (14 kB)
Using cached ddgs-9.11.4-py3-none-any.whl (43 kB)


In [1]:
import ollama
import requests
import yt_dlp
from bs4 import BeautifulSoup
from ddgs import DDGS
import time
import re
from IPython.display import display, Markdown

# --- CONFIG ---
MODEL = "llama3.2:1b"        # change to phi3 or phi if preferred
MAX_SEARCH_RESULTS = 6       # URLs per sub-query
MAX_SUB_QUERIES = 3          # sub-queries to generate
MAX_CHUNK_CHARS = 1500       # chars per scraped chunk
TOP_CHUNKS = 8               # top chunks for final synthesis
REQUEST_TIMEOUT = 10         # HTTP timeout in seconds
YT_MAX_RESULTS = 6           # YouTube videos to return

print(f"Model: {MODEL}  |  Ready.")

Model: llama3.2:1b  |  Ready.


## 2. Core Utilities

In [4]:
def llm(prompt: str, system: str = "", stream: bool = False) -> str:
    """Call Ollama synchronously."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    if stream:
        response = ""
        for chunk in ollama.chat(model=MODEL, messages=messages, stream=True):
            piece = chunk["message"]["content"]
            print(piece, end="", flush=True)
            response += piece
        print()
        return response
    else:
        result = ollama.chat(model=MODEL, messages=messages)
        return result["message"]["content"]


def search_web(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list[dict]:
    """Search DuckDuckGo and return list of {title, url, snippet}."""
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title": r.get("title", ""),
                    "url": r.get("href", ""),
                    "snippet": r.get("body", ""),
                })
    except Exception as e:
        print(f"  [Search error: {e}]")
    return results


def fetch_page(url: str) -> str:
    """Fetch and extract clean text from a URL."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; DeepSearch/1.0)"}
    try:
        resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # Remove boilerplate tags
        for tag in soup(["script", "style", "nav", "footer", "header",
                         "aside", "form", "iframe", "noscript"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
        # Collapse whitespace
        text = re.sub(r"\s+", " ", text)
        return text
    except Exception as e:
        return ""


def chunk_text(text: str, chunk_size: int = MAX_CHUNK_CHARS) -> list[str]:
    """Split text into overlapping chunks."""
    chunks = []
    step = int(chunk_size * 0.8)
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk) > 200:  # skip tiny chunks
            chunks.append(chunk)
    return chunks


print("Utilities defined.")

Utilities defined.


## 3. DeepSearch Pipeline

In [5]:
def generate_sub_queries(topic: str) -> list[str]:
    """Ask the LLM to generate targeted search queries for the topic."""
    print("[1/4] Generating sub-queries...")
    prompt = f"""Generate {MAX_SUB_QUERIES} specific, diverse search queries to thoroughly research:
"{topic}"

Return ONLY the queries, one per line, no numbering, no explanation."""
    raw = llm(prompt)
    queries = [q.strip() for q in raw.strip().splitlines() if q.strip()]
    queries = queries[:MAX_SUB_QUERIES]
    # Always include the original topic
    if topic not in queries:
        queries.insert(0, topic)
    print(f"  Queries: {queries}")
    return queries


def gather_sources(queries: list[str]) -> list[dict]:
    """Search and deduplicate URLs."""
    print("[2/4] Searching the web...")
    seen_urls = set()
    all_results = []
    for q in queries:
        results = search_web(q)
        for r in results:
            if r["url"] and r["url"] not in seen_urls:
                seen_urls.add(r["url"])
                all_results.append(r)
        time.sleep(0.5)  # polite delay
    print(f"  Found {len(all_results)} unique sources.")
    return all_results


def scrape_and_rank(sources: list[dict], topic: str) -> list[str]:
    """Fetch pages, chunk, and score relevance with LLM."""
    print("[3/4] Scraping pages & ranking chunks...")
    scored_chunks = []
    
    for i, src in enumerate(sources):
        url = src["url"]
        print(f"  [{i+1}/{len(sources)}] {url[:80]}")
        
        # Use snippet as fallback if scraping fails
        page_text = fetch_page(url)
        if not page_text:
            page_text = src.get("snippet", "")
        
        if not page_text:
            continue
        
        chunks = chunk_text(page_text)
        
        # Score each chunk for relevance
        for chunk in chunks[:4]:  # limit chunks per page to save time
            score_prompt = f"""Rate how relevant this text is to the topic "{topic}" on a scale 0-10.
Return ONLY a number.

Text: {chunk[:500]}"""
            try:
                score_str = llm(score_prompt).strip()
                score = float(re.search(r"\d+\.?\d*", score_str).group())
            except:
                score = 5.0
            
            scored_chunks.append((score, chunk, src["title"], url))
    
    # Sort by relevance descending
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    top = scored_chunks[:TOP_CHUNKS]
    
    print(f"  Selected {len(top)} top chunks (scores: {[round(s,1) for s,*_ in top]})")
    return top


def synthesize_report(topic: str, top_chunks: list) -> str:
    """Generate the final deep research report."""
    print("[4/4] Synthesizing report (streaming)...\n")
    
    context_parts = []
    for score, chunk, title, url in top_chunks:
        context_parts.append(f"--- Source: {title} ({url}) ---\n{chunk}")
    context = "\n\n".join(context_parts)
    
    system = """You are a thorough research assistant. Write a comprehensive, well-structured report 
based on the provided sources. Use markdown with headers, bullet points, and clear sections. 
Cite sources by title when referencing specific information. Be factual and detailed."""
    
    prompt = f"""Based on the following research sources, write a comprehensive deep-dive report on:
**{topic}**

SOURCES:
{context}

Write the report now:"""
    
    return llm(prompt, system=system, stream=True)


def deep_search(topic: str):
    """Main DeepSearch entry point."""
    print(f"\n{'='*60}")
    print(f"DeepSearch: {topic}")
    print(f"{'='*60}\n")
    
    queries = generate_sub_queries(topic)
    sources = gather_sources(queries)
    top_chunks = scrape_and_rank(sources, topic)
    report = synthesize_report(topic, top_chunks)
    
    print(f"\n{'='*60}")
    print("Done.")
    return report


print("Pipeline defined.")

Pipeline defined.


## 4. Run DeepSearch

Change the `TOPIC` variable and run the cell.

In [17]:
# ---- SET YOUR TOPIC HERE ----
TOPIC = "quantum computing recent breakthroughs 2024"
# -----------------------------

report = deep_search(TOPIC)


DeepSearch: quantum computing recent breakthroughs 2024

[1/4] Generating sub-queries...
  Queries: ['quantum computing recent breakthroughs 2024', 'What is quantum computing?', 'What are some of the key breakthroughs in quantum computing technology so far this year 2024', 'Which companies are leading the charge in developing practical applications for quantum computing and what are their strategies?']
[2/4] Searching the web...
  Found 24 unique sources.
[3/4] Scraping pages & ranking chunks...
  [1/24] https://fucoder.com/latest-breakthroughs-in-quantum-computing-2024/
  [2/24] https://willametteweekly.news/quantum-computing-breakthroughs-of-june-2024-a-new
  [3/24] https://techtobard.com/recent-advances-in-quantum-computing/
  [4/24] https://gimkit.blog/latest-breakthroughs-in-quantum-computing-2024/
  [5/24] https://techworlzupdates.com/how-quantum-computing-hardware-is-redefining-techno
  [6/24] https://www.weblogicnet.com/how-quantum-computing-hardware-is-redefining-technol
  [7

In [18]:
# Render the final report as formatted Markdown
display(Markdown(report))

**Quantum Computing Report**

**Introduction**

Quantum computing is a field of research that uses the principles of quantum mechanics to perform calculations and operations on data. In recent years, significant advancements have been made in this area, leading to the development of large-scale, error-corrected quantum computers. This report provides an overview of the current state of quantum computing, its capabilities, and its potential applications.

**Quantum Computing Basics**

Quantum computing is based on the concept of superposition, where a single qubit can exist in multiple states simultaneously. This property allows quantum computers to process information in parallel, making them potentially much faster than classical computers for certain tasks. Quantum computers also use entanglement, which enables the sharing of information between qubits.

**Applications and Advantages**

Quantum computing has several potential applications, including:

1. **Cryptography**: Quantum computers can break many classical encryption algorithms, but they can also be used to create new, quantum-resistant ones.
2. **Optimization**: Quantum computers can solve complex optimization problems more efficiently than classical computers.
3. **Simulations**: Quantum computers can simulate the behavior of molecules and other complex systems, leading to breakthroughs in fields like chemistry and materials science.

**Current State of Quantum Computing**

Several companies are working on building large-scale quantum computers, including:

1. **Google**: Google has developed several quantum chips, including the Sycamore processor, which demonstrated quantum supremacy.
2. **IBM**: IBM has developed a 53-qubit quantum computer called the IBM Q System One, which is available for use through the IBM Quantum Experience cloud platform.
3. **Rigetti Computing**: Rigetti Computing has developed a 128-qubit quantum computer that is being used to simulate complex systems.

**Error Correction and Scalability**

Error correction is a major challenge in building large-scale quantum computers. Researchers are developing new algorithms and techniques, such as error correction codes, to mitigate this issue. Scaling up the number of qubits while maintaining error correction is an ongoing challenge.

**Future Directions**

Several companies, including Google, IBM, and Rigetti Computing, are working towards developing practical, fault-tolerant quantum systems that can be used for real-world applications. The expected timeline for these developments is around 2030, although this may vary depending on the company's progress.

**Conclusion**

Quantum computing has made significant advancements in recent years, with companies like Google and IBM pushing the boundaries of what is possible. While there are still challenges to overcome, quantum computers have the potential to revolutionize many fields, from cryptography to optimization. As researchers continue to develop new technologies and techniques, we can expect to see more practical applications emerge.

**Recommendations**

1. **Investment**: Governments and private companies should invest in research and development of quantum computing.
2. **Standardization**: Standardization of quantum protocols and algorithms will facilitate the development of large-scale quantum computers.
3. **Education and Training**: Education and training programs will be necessary to develop a skilled workforce with expertise in quantum computing.

**Timeline**

* 2024: Quantum Embark Program launched by AWS, providing context and guidance for customers
* 2025-2030: Development of practical, fault-tolerant quantum systems expected

**Sources**

1. "What is Quantum Computing?" | OVHcloud Worldwide (https://www.ovhcloud.com/en/learn/what-is-quantum-computing/)
2. "12 companies building quantum computers" | TechTarget (https://www.techtarget.com/searchdatacenter/feature/Companies-building-quantum-computers)
3. "Latest Breakthroughs in Quantum Computing 2024 - Adventuretimes" (https://adventuretimes.co.uk/latest-breakthroughs-in-quantum-computing-2024/)

## 5. Quick Search (no scraping — fast mode)

In [19]:
def quick_search(topic: str):
    """Faster version: use DuckDuckGo snippets only, no full page scraping."""
    print(f"Quick search: {topic}\n")
    
    results = search_web(topic, max_results=10)
    snippets = "\n\n".join(
        f"**{r['title']}** ({r['url']})\n{r['snippet']}"
        for r in results if r['snippet']
    )
    
    system = "You are a research assistant. Summarize and synthesize the search results into a clear, structured answer."
    prompt = f"Topic: {topic}\n\nSearch results:\n{snippets}\n\nWrite a comprehensive summary:"
    
    return llm(prompt, system=system, stream=True)


# ---- SET YOUR TOPIC HERE ----
QUICK_TOPIC = "latest AI models 2025"
# -----------------------------

quick_report = quick_search(QUICK_TOPIC)
display(Markdown(quick_report))

Quick search: latest AI models 2025

**Summary of Latest AI Models in 2025**

The latest advancements in Artificial Intelligence (AI) in 2025 have been significant, with several notable releases and updates from major companies. Here's a comprehensive summary:

* **Google's AI News Recap 2025**: Google announced several exciting developments in its AI ecosystem, including updates to Gemini, the AI Mode in Search, and new hardware.
* **The Best AI Tools of 2025**: A practical guide by Ilampadmanabhan on the best AI tools for various tasks such as writing, marketing, productivity, coding, and automation. The top tools include Opus, Claude, Llama, and Gemini.
* **2025 LLM Year in Review**: A blog post by Karpathy on the 2025 LLM (Large Language Model) year in review, highlighting several advancements, including Reinforcement Learning from Verifiable Rewards (RLVR), Ghosts vs. Animals/Jagged Intelligence, and Cursor.
* **The 2025 AI Index Report**: Stanford HAI released its annual AI index

**Summary of Latest AI Models in 2025**

The latest advancements in Artificial Intelligence (AI) in 2025 have been significant, with several notable releases and updates from major companies. Here's a comprehensive summary:

* **Google's AI News Recap 2025**: Google announced several exciting developments in its AI ecosystem, including updates to Gemini, the AI Mode in Search, and new hardware.
* **The Best AI Tools of 2025**: A practical guide by Ilampadmanabhan on the best AI tools for various tasks such as writing, marketing, productivity, coding, and automation. The top tools include Opus, Claude, Llama, and Gemini.
* **2025 LLM Year in Review**: A blog post by Karpathy on the 2025 LLM (Large Language Model) year in review, highlighting several advancements, including Reinforcement Learning from Verifiable Rewards (RLVR), Ghosts vs. Animals/Jagged Intelligence, and Cursor.
* **The 2025 AI Index Report**: Stanford HAI released its annual AI index report, which shows open-weight models closing the gap with closed models, reducing performance differences from 8% to 1.7%.
* **AI Model Releases November/December 2025**: Four major AI companies launched their most powerful models yet: xAI's Grok 4.1 (November 17), Google's Gemini 3 (November 15), Claude 4.5, and GPT-... (no specific details).
* **Global AI Adoption in 2025**: A report by Microsoft on the rise of DeepSeek, a new AI entrant that surprised the industry with its flagship model.

**Key Findings:**

* Open-weight models are closing the gap with closed models.
* Four major AI companies launched their most powerful models yet in November and December 2025.
* Several new AI tools have been released, including Opus, Claude, Llama, and Gemini.
* The 2025 AI index report shows significant advancements in AI research.

**Conclusion:**

The latest developments in AI in 2025 demonstrate the rapid progress being made in this field. From Google's updates to OpenAI's new models, it's clear that the AI landscape is becoming increasingly sophisticated. As the field continues to evolve, we can expect even more exciting advancements and innovations from top companies like Google, Microsoft, and OpenAI.

## 6. YouTube Search — Video IDs & Metadata

In [1]:
print("YouTube section ready. Use search_youtube(query) below.")

YouTube section ready. Use search_youtube(query) below.


In [6]:
def search_youtube(query, max_results=YT_MAX_RESULTS):
    """Search YouTube without downloading. Returns video id, title, channel, duration, views, url, thumbnail."""
    ydl_opts = {"quiet": True, "no_warnings": True, "extract_flat": True}
    videos = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(f"ytsearch{max_results}:{query}", download=False)
        for entry in info.get("entries", []):
            d = int(entry.get("duration") or 0)
            vid_id = entry.get("id", "")
            videos.append({
                "id":       vid_id,
                "title":    entry.get("title", ""),
                "channel":  entry.get("channel") or entry.get("uploader", ""),
                "duration": f"{d // 60}:{d % 60:02d}",
                "views":    entry.get("view_count"),
                "desc":     (entry.get("description") or "")[:250],
                "url":      "https://www.youtube.com/watch?v=" + vid_id,
                "thumb":    "https://img.youtube.com/vi/" + vid_id + "/hqdefault.jpg",
            })
    return videos


print("search_youtube() ready.")

search_youtube() ready.


In [7]:
def display_youtube_results(videos):
    """Render results as a Markdown table + plain ID list."""
    if not videos:
        print("No results.")
        return
    rows = [
        "| # | Title | Channel | Duration | Views | ID |",
        "|---|-------|---------|----------|-------|----|",
    ]
    for i, v in enumerate(videos, 1):
        views = f"{v['views']:,}" if v["views"] else "N/A"
        title = f"[{v['title'][:55]}]({v['url']})"
        rows.append(f"| {i} | {title} | {v['channel']} | {v['duration']} | {views} | `{v['id']}` |")
    display(Markdown("\n".join(rows)))
    print("\nVideo IDs:")
    for v in videos:
        print(f"  {v['id']}   {v['url']}")
        if v["desc"]:
            print(f"  {v['desc'][:110]}")
        print()


print("display_youtube_results() ready.")

display_youtube_results() ready.


In [12]:
# ---- SET YOUR YOUTUBE QUERY HERE ----
YT_QUERY = "quantum computing explained 2024"
# --------------------------------------

yt_videos = search_youtube(YT_QUERY)
display_youtube_results(yt_videos)

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 2 | [Quantum Computing Explained in 2 Minutes 2024](https://www.youtube.com/watch?v=bEYK8gxuFqs) | @TechTomenia | 1:19 | 4 | `bEYK8gxuFqs` |
| 3 | [Quantum Computing Explained: 20 Ways It Will Affect EVE](https://www.youtube.com/watch?v=79kNOf749MA) | Future Business Tech | 86:29 | 346,505 | `79kNOf749MA` |
| 4 | [Brian Cox on quantum computing and black hole physics](https://www.youtube.com/watch?v=laGXRs9Ce70) | Big Think | 6:43 | 1,088,122 | `laGXRs9Ce70` |
| 5 | [Quantum Computing 2025 Update](https://www.youtube.com/watch?v=cDt7o-OmBWI) | ExplainingComputers | 17:05 | 61,948 | `cDt7o-OmBWI` |
| 6 | [How Physicists Proved Everything is Quantum - Nobel Phy](https://www.youtube.com/watch?v=_mVBbdbqHmw) | Dr Ben Miles | 7:18 | 1,295,422 | `_mVBbdbqHmw` |


Video IDs:
  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  bEYK8gxuFqs   https://www.youtube.com/watch?v=bEYK8gxuFqs
  Quantum computing is a revolutionary field that leverages the principles of quantum mechanics to process infor

  79kNOf749MA   https://www.youtube.com/watch?v=79kNOf749MA
  Boost your knowledge in quantum computing and other emerging technologies with Brilliant's engaging courses. E

  laGXRs9Ce70   https://www.youtube.com/watch?v=laGXRs9Ce70
  Become a Big Think member to unlock expert classes, premium print issues, exclusive events and more: ...

  cDt7o-OmBWI   https://www.youtube.com/watch?v=cDt7o-OmBWI
  Quantum computing developments over the past 12 months, including Google Willow, IBM Majorana 1, and innovatio

  _mVBbdbqHmw   https://www.youtube.com/watch?v=_mVBbdbqHmw
  The Nobel Prize in Physics 2025 was awarded jointly to John Cl

## 7. Combined: Web Report + YouTube Videos

In [22]:
def full_research(topic):
    """Quick web summary + YouTube videos for any topic."""
    print(f"\n{'='*60}\nWEB: {topic}\n{'='*60}")
    report = quick_search(topic)
    print(f"\n{'='*60}\nYOUTUBE: {topic}\n{'='*60}")
    videos = search_youtube(topic)
    display_youtube_results(videos)
    return report, videos


# ---- SET TOPIC ----
COMBINED_TOPIC = "large language models 2025"
# -------------------

web_report, videos = full_research(COMBINED_TOPIC)
display(Markdown(web_report))


WEB: large language models 2025
Quick search: large language models 2025

**Summary of Large Language Models in 2025**

Large language models (LLMs) have made significant progress in recent years, with advancements in various aspects of their design, training, and deployment. According to search results from reputable sources such as Wikipedia, Magazine Sebastian Raschka, RYColab, Medium, Instaclustr, Superannotate, American CSE, and Machine Learning Mastery, here is a comprehensive summary of the current state and future outlook of LLMs in 2025:

**Current State:**

* Large models are language models or multimodal models with language capacity.
* The majority of large models are trained on large-scale datasets and consist of multiple layers.
* Recent developments include the training of model variants such as o3, DeepSeek R1, RLVR, and inference-time scaling.
* Benchmarks, architectures, and predictions for LLMs are continually being updated to improve performance.

**Predictions for

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [How to Choose Large Language Models: A Developer’s Guid](https://www.youtube.com/watch?v=pYax2rupKEY) | IBM Technology | 6:57 | 94,681 | `pYax2rupKEY` |
| 2 | [Large Language Models explained briefly](https://www.youtube.com/watch?v=LPZh9BOjkQs) | 3Blue1Brown | 7:58 | 5,524,580 | `LPZh9BOjkQs` |
| 3 | [Inside GPT – Large Language Models Demystified - Alan S](https://www.youtube.com/watch?v=f6dFD1Hzv20) | NDC Conferences | 61:28 | 2,339 | `f6dFD1Hzv20` |
| 4 | [How Large Language Models Work](https://www.youtube.com/watch?v=5sLYAQS9sWQ) | IBM Technology | 5:34 | 1,397,965 | `5sLYAQS9sWQ` |
| 5 | [Large Language Models (LLMs) (2025)](https://www.youtube.com/watch?v=yToIlnqNu1I) | 10 mins learning | 8:25 | 46 | `yToIlnqNu1I` |
| 6 | [Christopher Manning: Large Language Models in 2025 – Ho](https://www.youtube.com/watch?v=5Aer7MUSuSU) | Stanford HAI | 39:23 | 18,410 | `5Aer7MUSuSU` |


Video IDs:
  pYax2rupKEY   https://www.youtube.com/watch?v=pYax2rupKEY
  Ready to become a certified watsonx AI Assistant Engineer? Register now and use code IBMTechYT20 for 20% off o

  LPZh9BOjkQs   https://www.youtube.com/watch?v=LPZh9BOjkQs
  A light intro to LLMs, chatbots, pretraining, and transformers. Dig deeper here: ...

  f6dFD1Hzv20   https://www.youtube.com/watch?v=f6dFD1Hzv20
  This talk was recorded at NDC AI in Oslo, Norway. #ndcai #ndcconferences #developer #softwaredeveloper Attend 

  5sLYAQS9sWQ   https://www.youtube.com/watch?v=5sLYAQS9sWQ
  Learn in-demand Machine Learning skills now → https://ibm.biz/BdK65D Learn about watsonx → https://ibm.biz/Bdv

  yToIlnqNu1I   https://www.youtube.com/watch?v=yToIlnqNu1I
  Large Language Models (LLMs) technology, particularly academic papers, has outlined the architecture, construc

  5Aer7MUSuSU   https://www.youtube.com/watch?v=5Aer7MUSuSU
  The Stanford Open Virtual Assistant Lab, with sponsorship from the Alfred P. Sloan

**Summary of Large Language Models in 2025**

Large language models (LLMs) have made significant progress in recent years, with advancements in various aspects of their design, training, and deployment. According to search results from reputable sources such as Wikipedia, Magazine Sebastian Raschka, RYColab, Medium, Instaclustr, Superannotate, American CSE, and Machine Learning Mastery, here is a comprehensive summary of the current state and future outlook of LLMs in 2025:

**Current State:**

* Large models are language models or multimodal models with language capacity.
* The majority of large models are trained on large-scale datasets and consist of multiple layers.
* Recent developments include the training of model variants such as o3, DeepSeek R1, RLVR, and inference-time scaling.
* Benchmarks, architectures, and predictions for LLMs are continually being updated to improve performance.

**Predictions for 2025:**

* The development of new models with improved language understanding and generation capabilities.
* Advancements in training methods, such as model parallelization and distributed training.
* The integration of multiple AI technologies, including reinforcement learning and transfer learning.
* The emergence of new architectures, such as transformer-based models and attention mechanisms.

**Top 10 Open Source LLMs for 2025:**

* These open-source LLMs are machine learning models that can understand and generate human language based on large-scale datasets.

**Fine-tuning Large Language Models in 2025:**

* The importance of fine-tuning LLMs to adapt them to specific tasks, domains, or industries.
* Methods and best practices for optimizing language model performance, including data preprocessing, training architectures, and hyperparameter tuning.

**LLM 2025 Conference:**

* This conference aims to provide a platform to exchange ideas, share recent findings, explore use cases, and engage in discussions on the latest advancements in LLMs.
* The conference will focus on topics such as language understanding, generation, and application of LLMs in various domains.

**The Ultimate Guide to Top Large Language Models in 2025:**

* This guide provides an overview of the top trending LLMs in 2025, including GPT-5 (OpenAI), Gemini, Command R+, and LLaMA 4.
* It covers topics such as architectures, training methods, and applications of these models.

**The Roadmap for Mastering Language Models in 2025:**

* This roadmap provides a step-by-step guide to mastering language models in 2025, covering topics such as fundamental understanding, core architecture, specializations, and knowledge absorption.
* The roadmap emphasizes the importance of continuous learning, exploration, and adaptation of LLMs to remain competitive in the AI landscape.

**Teaching Large Language Models How to Absorb New Knowledge:**

* Researchers have developed techniques for teaching large language models to absorb new knowledge by generating study sheets based on data.
* This approach enables LLMs to permanently incorporate new information into their training datasets, improving their overall performance and adaptability.

In [ ]:
## 8. PostgreSQL Storage (Docker)\n\nAll searches, results and reports are auto-saved.\n\n**Start the container once:**\n```bash\ndocker run -d --name deepsearch-pg \\\n  -e POSTGRES_USER=deepsearch \\\n  -e POSTGRES_PASSWORD=deepsearch \\\n  -e POSTGRES_DB=deepsearch \\\n  -p 5432:5432 postgres:16-alpine\n```

In [29]:
!pip install psycopg2-binary SQLAlchemy -q

In [23]:
!docker start deepsearch-pg


deepsearch-pg


In [19]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB = dict(host="localhost", port=5432, dbname="deepsearch", user="deepsearch", password="deepsearch")

def get_conn():
    return psycopg2.connect(**DB)

def init_db():
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS searches (
                    id          SERIAL PRIMARY KEY,
                    topic       TEXT NOT NULL,
                    search_type TEXT NOT NULL,
                    created_at  TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS web_results (
                    id        SERIAL PRIMARY KEY,
                    search_id INT REFERENCES searches(id) ON DELETE CASCADE,
                    title     TEXT,
                    url       TEXT,
                    snippet   TEXT,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS youtube_results (
                    id          SERIAL PRIMARY KEY,
                    search_id   INT REFERENCES searches(id) ON DELETE CASCADE,
                    video_id    TEXT,
                    title       TEXT,
                    channel     TEXT,
                    duration    TEXT,
                    views       BIGINT,
                    description TEXT,
                    url         TEXT,
                    thumbnail   TEXT,
                    created_at  TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS reports (
                    id        SERIAL PRIMARY KEY,
                    search_id INT REFERENCES searches(id) ON DELETE CASCADE,
                    content   TEXT,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
            """)
        conn.commit()
    print("DB ready — tables: searches, web_results, youtube_results, reports")

init_db()

DB ready — tables: searches, web_results, youtube_results, reports


In [20]:
def _new_search(topic, kind):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("INSERT INTO searches (topic, search_type) VALUES (%s,%s) RETURNING id", (topic, kind))
            sid = cur.fetchone()[0]
        conn.commit()
    return sid

def _save_web(sid, results):
    if not results: return
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.executemany(
                "INSERT INTO web_results (search_id,title,url,snippet) VALUES (%s,%s,%s,%s)",
                [(sid, r["title"], r["url"], r["snippet"]) for r in results]
            )
        conn.commit()

def _save_yt(sid, videos):
    if not videos: return
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.executemany(
                """INSERT INTO youtube_results
                   (search_id,video_id,title,channel,duration,views,description,url,thumbnail)
                   VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                [(sid, v["id"], v["title"], v["channel"], v["duration"],
                  v["views"], v["desc"], v["url"], v["thumb"]) for v in videos]
            )
        conn.commit()

def _save_report(sid, text):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("INSERT INTO reports (search_id,content) VALUES (%s,%s)", (sid, text))
        conn.commit()

print("DB helpers ready.")

DB helpers ready.


In [21]:
def db_quick_search(topic):
    """Quick web search + save to DB."""
    sid = _new_search(topic, "quick")
    results = search_web(topic, max_results=10)
    _save_web(sid, results)
    snippets = "\n\n".join(f"**{r['title']}**\n{r['snippet']}" for r in results if r["snippet"])
    report = llm(f"Topic: {topic}\n\nResults:\n{snippets}\n\nSummary:",
                 system="Summarize into a clear structured answer.", stream=True)
    _save_report(sid, report)
    print(f"\n[DB] search_id={sid} | {len(results)} web results + report saved")
    return sid, report

def db_youtube_search(topic):
    """YouTube search + save to DB."""
    sid = _new_search(topic, "youtube")
    videos = search_youtube(topic)
    _save_yt(sid, videos)
    print(f"[DB] search_id={sid} | {len(videos)} YouTube videos saved")
    display_youtube_results(videos)
    return sid, videos

def db_full_research(topic):
    """Web + YouTube + report — everything saved to DB."""
    sid = _new_search(topic, "combined")
    # Web
    results = search_web(topic, max_results=10)
    _save_web(sid, results)
    snippets = "\n\n".join(f"**{r['title']}**\n{r['snippet']}" for r in results if r["snippet"])
    report = llm(f"Topic: {topic}\n\nResults:\n{snippets}\n\nSummary:",
                 system="Summarize into a clear structured answer.", stream=True)
    _save_report(sid, report)
    # YouTube
    videos = search_youtube(topic)
    _save_yt(sid, videos)
    print(f"\n[DB] search_id={sid} | {len(results)} web + {len(videos)} YouTube + report saved")
    display_youtube_results(videos)
    display(Markdown(report))
    return sid

print("DB search functions ready.")

DB search functions ready.


In [22]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn physics and electromagnetism"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear and structured answer to your question:

**Learning Physics and Electromagnetism**

To learn physics and electromagnetism, it's essential to start with the basics. Here are some steps you can follow:

1. **Understand the concepts**: Start by understanding the fundamental concepts of physics, such as motion, energy, and forces. You can use online resources like [The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism](https://www.feynmanlectures.net/) to get a grasp on these ideas.
2. **Find a suitable textbook**: Once you have a good understanding of the concepts, find a textbook that covers electromagnetism and electricity. Some recommended textbooks include [AP Physics C: Electricity and Magnetism - AP Students](https://www Courseasy.com/learn/physics/ap-physics-c-electricity-and-magnetism/) and [How to Learn Physics for Free](https://www.apphysicsc.com/physicist/learn).
3. **Watch online resources**: There are many online resources available to learn electroma

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [6 Books to Self-Teach Electromagnetic Physics](https://www.youtube.com/watch?v=S6Q2Da2Ku8E) | Ali the Dazzling | 7:23 | 67,368 | `S6Q2Da2Ku8E` |
| 2 | [Electromagnetism Explained in Simple Words](https://www.youtube.com/watch?v=nllCgjlWAF4) | Science ABC | 4:14 | 283,319 | `nllCgjlWAF4` |
| 3 | [Why Physics Is Hard](https://www.youtube.com/watch?v=8fjWafOajjc) | Professor Hafner | 2:37 | 738,903 | `8fjWafOajjc` |
| 4 | [Want to study physics? Read these 10 books](https://www.youtube.com/watch?v=p9s2fBYA4fU) | Simon Clark | 14:16 | 2,286,423 | `p9s2fBYA4fU` |
| 5 | [How To Study Hard - Richard Feynman](https://www.youtube.com/watch?v=YDV1mo7QlnA) | Arjun Kocher | 3:19 | 4,732,839 | `YDV1mo7QlnA` |
| 6 | [Physics for Absolute Beginners](https://www.youtube.com/watch?v=iE-M56CVJx4) | The Math Sorcerer | 13:06 | 386,835 | `iE-M56CVJx4` |


Video IDs:
  S6Q2Da2Ku8E   https://www.youtube.com/watch?v=S6Q2Da2Ku8E
  Electromagnetic physics is the most important discipline to understand for electrical engineering students. Sa

  nllCgjlWAF4   https://www.youtube.com/watch?v=nllCgjlWAF4
  Electromagnetism is a branch of physics that deals with the study of electromagnetic forces, including electri

  8fjWafOajjc   https://www.youtube.com/watch?v=8fjWafOajjc
  This is an intro video from my online classes.

  p9s2fBYA4fU   https://www.youtube.com/watch?v=p9s2fBYA4fU
  Books for physics students! Popular science books and textbooks to get you from high school to university. Als

  YDV1mo7QlnA   https://www.youtube.com/watch?v=YDV1mo7QlnA
  Study hard what interests you the most in the most undisciplined, irreverent and original manner possible. - R

  iE-M56CVJx4   https://www.youtube.com/watch?v=iE-M56CVJx4
  This video will show you some books you can use to help get started with physics. Do you have any other recomm



Here's a clear and structured answer to your question:

**Learning Physics and Electromagnetism**

To learn physics and electromagnetism, it's essential to start with the basics. Here are some steps you can follow:

1. **Understand the concepts**: Start by understanding the fundamental concepts of physics, such as motion, energy, and forces. You can use online resources like [The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism](https://www.feynmanlectures.net/) to get a grasp on these ideas.
2. **Find a suitable textbook**: Once you have a good understanding of the concepts, find a textbook that covers electromagnetism and electricity. Some recommended textbooks include [AP Physics C: Electricity and Magnetism - AP Students](https://www Courseasy.com/learn/physics/ap-physics-c-electricity-and-magnetism/) and [How to Learn Physics for Free](https://www.apphysicsc.com/physicist/learn).
3. **Watch online resources**: There are many online resources available to learn electromagnetism, including videos, tutorials, and interactive courses. Some recommended resources include [Magnetism and Electromagnetism | AP/Crash Course Physics 2](https://www.youtube.com/watch?v=1xY6Bf7yHmQ) and [Learn Electromagnetism - Free Interactive Course - Courseasy](https://courseasy.com/learn/electromagnetism).
4. **Join a community**: Joining a community of physics students or enthusiasts can be very helpful in learning electromagnetism. You can find online forums, social media groups, or Reddit communities dedicated to physics and electromagnetism.
5. **Practice, practice, practice**: Practice is key when it comes to learning physics and electromagnetism. Start with simple problems and gradually move on to more complex ones.

**Additional Tips**

* Make sure you have a good understanding of math concepts before diving into electromagnetism.
* Use online resources like Khan Academy or MIT OpenCourseWare for additional help.
* Don't be afraid to ask questions or seek help from your teachers or mentors.
* Stay motivated and patient - learning physics and electromagnetism takes time and effort.

By following these steps, you can learn physics and electromagnetism in a structured and effective way.


All data stored under search_id = 5


In [23]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn physics and electromagnetism give me resources and videos"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear and structured summary of resources and videos to learn electricity and electromagnetism:

**Online Courses and Lectures**

1. MIT OpenCourseWare: "Physics II: Electricity and Magnetism"
2. IOPSpark (Unlimited Access to Physics Teaching Resources): Remote teaching resources for electricity and magnetism
3. MITx: Maxwell's Equations, electromagnetic radiation, and electromagnetic fields

**Videos**

1. The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism ( online edition)
2. AP Physics C: Electricity and Magnetism Full Course/Full Walkthrough (YouTube)
3. Khan Academy videos on electricity and magnetism
4. JoVE's science videos on electromagnetism

**Textbooks and Resources**

1. MIT OpenCourseWare textbook: Maxwell’s equations, in addition to describing this behavior, also describe electromagnetic radiation.
2. Becker/Sauter edition of Abraham/Becker (follow-up edition) for advanced undergraduate level study
3. Dover book: "Physics: Tutoring, Video Lessons, Co

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [ELECTROMAGNETISM for Kids⚡🧲 What are Electromagnets? 🔌 ](https://www.youtube.com/watch?v=tHIchO1pbFA) | Smile and Learn - English | 4:00 | 216,517 | `tHIchO1pbFA` |
| 2 | [Teach Yourself Physics from SCRATCH. | Foundations 1.1 ](https://www.youtube.com/watch?v=4c02Njfh2Zk) | Mike's Channel | 4:43 | 60,719 | `4c02Njfh2Zk` |
| 3 | [What is Light? Maxwell and the Electromagnetic Spectrum](https://www.youtube.com/watch?v=pj_ya0e20vE) | Professor Dave Explains | 3:56 | 1,384,639 | `pj_ya0e20vE` |
| 4 | [Physics for Absolute Beginners](https://www.youtube.com/watch?v=iE-M56CVJx4) | The Math Sorcerer | 13:06 | 386,835 | `iE-M56CVJx4` |
| 5 | [Electricity for Kids | What is Electricity? Where does ](https://www.youtube.com/watch?v=Dx3RpXdJw2k) | Learn Bright | 13:54 | 3,810,675 | `Dx3RpXdJw2k` |
| 6 | [Want to study physics? Read these 10 books](https://www.youtube.com/watch?v=p9s2fBYA4fU) | Simon Clark | 14:16 | 2,286,423 | `p9s2fBYA4fU` |


Video IDs:
  tHIchO1pbFA   https://www.youtube.com/watch?v=tHIchO1pbFA
  Educational video for children that talks about electromagnetism and electromagnets. Electricity is a physical

  4c02Njfh2Zk   https://www.youtube.com/watch?v=4c02Njfh2Zk
  Beyond belief so what I want you to do in this course is follow with me this is a textbook called physics by c

  pj_ya0e20vE   https://www.youtube.com/watch?v=pj_ya0e20vE
  Up until a couple centuries ago, we had no idea what light is. It seems like magic, no? But there is no magic 

  iE-M56CVJx4   https://www.youtube.com/watch?v=iE-M56CVJx4
  This video will show you some books you can use to help get started with physics. Do you have any other recomm

  Dx3RpXdJw2k   https://www.youtube.com/watch?v=Dx3RpXdJw2k
  NOTE: We would like to correct an error in this video. Birds do not get electrocuted when resting on power lin

  p9s2fBYA4fU   https://www.youtube.com/watch?v=p9s2fBYA4fU
  Books for physics students! Popular science books and te

Here's a clear and structured summary of resources and videos to learn electricity and electromagnetism:

**Online Courses and Lectures**

1. MIT OpenCourseWare: "Physics II: Electricity and Magnetism"
2. IOPSpark (Unlimited Access to Physics Teaching Resources): Remote teaching resources for electricity and magnetism
3. MITx: Maxwell's Equations, electromagnetic radiation, and electromagnetic fields

**Videos**

1. The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism ( online edition)
2. AP Physics C: Electricity and Magnetism Full Course/Full Walkthrough (YouTube)
3. Khan Academy videos on electricity and magnetism
4. JoVE's science videos on electromagnetism

**Textbooks and Resources**

1. MIT OpenCourseWare textbook: Maxwell’s equations, in addition to describing this behavior, also describe electromagnetic radiation.
2. Becker/Sauter edition of Abraham/Becker (follow-up edition) for advanced undergraduate level study
3. Dover book: "Physics: Tutoring, Video Lessons, Courses, Problems & Practice" for AP Physics C: E and M skills

**Websites and Forums**

1. Physics Forums for self-study and discussion on electricity and magnetism
2. Khan Academy website with various electromagnetism resources


All data stored under search_id = 6


### Browse stored data

In [17]:
def db_history(limit=20):
    """Show all past searches."""
    with get_conn() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("""
                SELECT s.id, s.topic, s.search_type, s.created_at,
                       COUNT(DISTINCT w.id) AS web_count,
                       COUNT(DISTINCT y.id) AS yt_count,
                       COUNT(DISTINCT r.id) AS report_count
                FROM searches s
                LEFT JOIN web_results    w ON w.search_id = s.id
                LEFT JOIN youtube_results y ON y.search_id = s.id
                LEFT JOIN reports        r ON r.search_id = s.id
                GROUP BY s.id ORDER BY s.created_at DESC LIMIT %s
            """, (limit,))
            rows = cur.fetchall()
    if not rows:
        print("No searches stored yet.")
        return []
    headers = "| id | topic | type | web | yt | reports | created |"
    sep     = "|----|-------|------|-----|----|---------|---------|"
    lines = [headers, sep]
    for r in rows:
        ts = r["created_at"].strftime("%Y-%m-%d %H:%M")
        lines.append(f"| {r['id']} | {r['topic'][:40]} | {r['search_type']} | {r['web_count']} | {r['yt_count']} | {r['report_count']} | {ts} |")
    display(Markdown("\n".join(lines)))
    return rows

def db_get_report(search_id):
    """Retrieve and display a stored report."""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT content FROM reports WHERE search_id=%s ORDER BY id DESC LIMIT 1", (search_id,))
            row = cur.fetchone()
    if row:
        display(Markdown(row[0]))
    else:
        print(f"No report found for search_id={search_id}")

def db_get_youtube(search_id):
    """Retrieve and display stored YouTube results."""
    with get_conn() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("SELECT * FROM youtube_results WHERE search_id=%s", (search_id,))
            rows = cur.fetchall()
    if not rows:
        print(f"No YouTube results for search_id={search_id}")
        return
    videos = [dict(r) for r in rows]
    # remap keys for display_youtube_results
    for v in videos:
        v["id"]    = v["video_id"]
        v["desc"]  = v.get("description", "")
        v["thumb"] = v.get("thumbnail", "")
    display_youtube_results(videos)

print("DB viewer functions ready.")

DB viewer functions ready.


In [18]:
# View all past searches
db_history()

NameError: name 'get_conn' is not defined

In [35]:
# Load a saved report by search_id
db_get_report(1)

**Quantum Computing in 2024: Breakthroughs, Challenges, and What's Ahead**

Quantum computing has made significant progress in 2023, but as we move into 2024, several challenges must be addressed to realize its full potential. Here's a summary of the current state of quantum computing:

**Breakthroughs:**

1. **Advancements in Hardware:** IBM and other companies have developed new hardware designs that improve the performance and efficiency of quantum computers.
2. **Software Evolution:** Quantum algorithms are being developed and optimized for practical applications, such as cryptography and optimization problems.
3. **Cybersecurity Impacts:** Quantum computing has the potential to revolutionize cybersecurity by breaking certain types of encryption.

**Challenges:**

1. **Scalability:** Current quantum computers are not yet scalable enough to solve complex problems efficiently.
2. **Noise and Error Correction:** Maintaining control over quantum fluctuations (noise) is a significant challenge, and developing robust error correction methods is essential.
3. **Interpretation of Results:** Quantum computing generates vast amounts of data, which must be interpreted correctly for practical applications.

**Future Trends:**

1. **Quantum-AI Symbiosis:** Researchers are exploring the integration of quantum computers with artificial intelligence (AI) to drive advancements in fields like optimization and machine learning.
2. **Commercial Adoption:** Expect significant commercial adoption of quantum computing in 2024, driven by increased investment, partnerships, and public awareness.
3. **Roadmaps and Development Paths:** Multiple players are announcing new development paths or updating existing ones, solidifying the year's roadmap for quantum computing.

**Key Players:**

1. IBM
2. Google (Bostock Lab)
3. Rigetti Computing
4. Microsoft Quantum Development Kit

In summary, 2024 promises to be a transformative year for quantum computing, with significant breakthroughs in hardware, software, and challenges that must be addressed. As the field continues to evolve, expect exciting developments in areas like AI symbiosis, commercial adoption, and scalability.

In [36]:
# Load saved YouTube videos by search_id
db_get_youtube(1)

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computing 2024 Update](https://www.youtube.com/watch?v=tIJuyb5DWUw) | ExplainingComputers | 15:41 | 92,343 | `tIJuyb5DWUw` |
| 2 | [Companies, countries battle to develop quantum computer](https://www.youtube.com/watch?v=K4ssT6Dzmnw) | 60 Minutes | 13:15 | 2,493,889 | `K4ssT6Dzmnw` |
| 3 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 4 | [Quantum Computing: Where We Are and Where We’re Headed ](https://www.youtube.com/watch?v=9XB-LsfpvCU) | NVIDIA Developer | 123:20 | 230,803 | `9XB-LsfpvCU` |
| 5 | [Quantum Computing 2025 Update](https://www.youtube.com/watch?v=cDt7o-OmBWI) | ExplainingComputers | 17:05 | 61,948 | `cDt7o-OmBWI` |
| 6 | [Michio Kaku: Quantum computing is the next revolution](https://www.youtube.com/watch?v=qQviI1d_hFA) | Big Think | 11:18 | 3,350,527 | `qQviI1d_hFA` |


Video IDs:
  tIJuyb5DWUw   https://www.youtube.com/watch?v=tIJuyb5DWUw
  Quantum computing developments over the past 12 months, including progress towards fault-tolerant hardware, an

  K4ssT6Dzmnw   https://www.youtube.com/watch?v=K4ssT6Dzmnw
  Companies and countries are in a race to develop quantum computers. The machines could revolutionize problem-s

  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  9XB-LsfpvCU   https://www.youtube.com/watch?v=9XB-LsfpvCU
  NVIDIA founder and CEO Jensen Huang hosts industry leaders from Alice & Bob, Atom Computing, AWS, D-Wave, Infl

  cDt7o-OmBWI   https://www.youtube.com/watch?v=cDt7o-OmBWI
  Quantum computing developments over the past 12 months, including Google Willow, IBM Majorana 1, and innovatio

  qQviI1d_hFA   https://www.youtube.com/watch?v=qQviI1d_hFA
  Become a Big Think member to unlock expert classes, prem

In [38]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn about quantum computing breakthroughs in 2024"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

**Quantum Computing Breakthroughs in 2024**

In 2024, significant advancements have been made in quantum computing, transforming the field from theory to tangible progress. Some of the key breakthroughs include:

1. **Improved Quantum Engineering**: Researchers have developed new methods for improving quantum systems, enabling more stable and reliable quantum computing.
2. **Logical Qubits**: Breakthroughs have led to improvements in logical qubits, which are the fundamental building blocks of quantum computers. These improvements have increased the reliability and error correction capabilities of quantum computers.
3. **Noise Resistance**: Researchers have demonstrated significant improvements in noise resistance, enabling quantum computers to operate more reliably in real-world settings.

**Key Developments:**

* Google's Willow chip
* 50 logical qubits
* Advances in quantum error correction

**Hybrid Approaches**: Research groups have explored the use of hybrid approaches that combi

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 2 | [Biggest Breakthroughs in Computer Science: 2024](https://www.youtube.com/watch?v=fTMMsreAqX0) | Quanta Magazine | 10:47 | 527,759 | `fTMMsreAqX0` |
| 3 | [Quantum Computing Breakthroughs of 2024: 10 Ways It Wil](https://www.youtube.com/watch?v=RtYchdRl7Bo) | SoSF | 24:17 | 204 | `RtYchdRl7Bo` |
| 4 | [Michio Kaku: Quantum computing is the next revolution](https://www.youtube.com/watch?v=qQviI1d_hFA) | Big Think | 11:18 | 3,350,527 | `qQviI1d_hFA` |
| 5 | [Decoding the Universe: Quantum | Full Documentary | NOV](https://www.youtube.com/watch?v=t06aTX9jM34) | NOVA PBS Official | 53:58 | 7,486,513 | `t06aTX9jM34` |
| 6 | [New Quantum Physics Discoveries 2024](https://www.youtube.com/watch?v=g5qKExdt4wo) | SpaceXVideos | 19:40 | 1,157 | `g5qKExdt4wo` |


Video IDs:
  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  fTMMsreAqX0   https://www.youtube.com/watch?v=fTMMsreAqX0
  The year's biggest breakthroughs in computer science included a new understanding of what's going on in large 

  RtYchdRl7Bo   https://www.youtube.com/watch?v=RtYchdRl7Bo
  Voice and cover generated by AI The drafting and researching are done by me, but with AI involvement Video edi

  qQviI1d_hFA   https://www.youtube.com/watch?v=qQviI1d_hFA
  Become a Big Think member to unlock expert classes, premium print issues, exclusive events and more: ...

  t06aTX9jM34   https://www.youtube.com/watch?v=t06aTX9jM34
  Dive into the universe at the tiniest – and weirdest – of scales. Official Website: https://to.pbs.org/3CkDYDR

  g5qKExdt4wo   https://www.youtube.com/watch?v=g5qKExdt4wo
  Join us on a fascinating journey through the world of quantum 

**Quantum Computing Breakthroughs in 2024**

In 2024, significant advancements have been made in quantum computing, transforming the field from theory to tangible progress. Some of the key breakthroughs include:

1. **Improved Quantum Engineering**: Researchers have developed new methods for improving quantum systems, enabling more stable and reliable quantum computing.
2. **Logical Qubits**: Breakthroughs have led to improvements in logical qubits, which are the fundamental building blocks of quantum computers. These improvements have increased the reliability and error correction capabilities of quantum computers.
3. **Noise Resistance**: Researchers have demonstrated significant improvements in noise resistance, enabling quantum computers to operate more reliably in real-world settings.

**Key Developments:**

* Google's Willow chip
* 50 logical qubits
* Advances in quantum error correction

**Hybrid Approaches**: Research groups have explored the use of hybrid approaches that combine classical computing, artificial intelligence (AI), and quantum hardware to tackle complex problems more effectively.

**Future Predictions:**

* Google's Willow processor expected to be operational by 2025
* Harvard's 48-logical-qubit processor expected to be operational in 2026

**Industry Implications:**

* Quantum computers may potentially break current encryption methods, necessitating advancements in cryptographic techniques.
* Quantum computers have the potential to revolutionize various fields, including medical science, machine learning (ML), environmental sciences, and other areas of advancement.

Overall, 2024 has seen significant progress in quantum computing, with breakthroughs that are transforming the field from theory to tangible progress. As research continues to advance, we can expect even more exciting developments in the years to come.


All data stored under search_id = 3


In [39]:
db_history()

| id | topic | type | web | yt | reports | created |
|----|-------|------|-----|----|---------|---------|
| 3 | i want to learn about quantum computing  | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 2 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 1 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:31 |

[RealDictRow([('id', 3),
              ('topic',
               'i want to learn about quantum computing breakthroughs in 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 44, 35126, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 2),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 20, 763229, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 1),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 31, 59, 289279, tzinfo=datetime.timezone.utc)),
          

In [40]:
DB_TOPIC = "i want to learn about Machine Learning"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear structured answer on "How To Learn Machine Learning in 2025" based on the provided resources:

**Step-by-Step Guide to Learning Machine Learning**

1. **Prerequisites**: Start with essential mathematical and programming concepts, such as linear algebra, calculus, probability, and computer science fundamentals.
2. **Setup Environment**: Install necessary tools and libraries, including Python, scikit-learn, TensorFlow, or PyTorch.
3. **Key Concepts**: Learn about machine learning algorithms, data preprocessing, model evaluation, and interpretation.
4. **Algorithms**: Study popular algorithms like linear regression, decision trees, clustering, and neural networks.
5. **Real-World Projects**: Participate in machine learning competitions, build projects, or work on personal initiatives to apply concepts learned.
6. **Certification Programs**: Consider obtaining certifications from organizations like Google, IBM, or edX to demonstrate expertise.

**Recommended Resources**

* *

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Machine Learning Explained in 100 Seconds](https://www.youtube.com/watch?v=PeMlggyqz0Y) | Fireship | 2:35 | 1,020,680 | `PeMlggyqz0Y` |
| 2 | [Machine Learning | What Is Machine Learning? | Introduc](https://www.youtube.com/watch?v=ukzFI9rgwfU) | Simplilearn | 7:52 | 5,372,393 | `ukzFI9rgwfU` |
| 3 | [The Complete Machine Learning Roadmap](https://www.youtube.com/watch?v=7IgVGSaQPaw) | Programming with Mosh | 5:25 | 944,331 | `7IgVGSaQPaw` |
| 4 | [Learn Machine Learning Like a GENIUS and Not Waste Time](https://www.youtube.com/watch?v=qNxrPri1V0I) | Infinite Codes | 15:03 | 1,193,951 | `qNxrPri1V0I` |
| 5 | [Learn Machine Learning Like a GENIUS and Not Waste Time](https://www.youtube.com/watch?v=2wTRQqoDM1A) | Sahil & Sarra | 10:33 | 288,833 | `2wTRQqoDM1A` |
| 6 | [Machine Learning is Probably Not a Good Career for You](https://www.youtube.com/watch?v=-8qzzN7CWxA) | the data janitor | 1:38 | 119,037 | `-8qzzN7CWxA` |


Video IDs:
  PeMlggyqz0Y   https://www.youtube.com/watch?v=PeMlggyqz0Y
  Machine Learning is the process of teaching a computer how perform a task with out explicitly programming it. 

  ukzFI9rgwfU   https://www.youtube.com/watch?v=ukzFI9rgwfU
  "️   Michigan Engineering - Professional Certificate in AI and Machine Learning ...

  7IgVGSaQPaw   https://www.youtube.com/watch?v=7IgVGSaQPaw
  Go from zero to a machine learning engineer in 12 months. This step-by-step roadmap covers the essential skill

  qNxrPri1V0I   https://www.youtube.com/watch?v=qNxrPri1V0I
  Learn Machine Learning Like a GENIUS and Not Waste Time ######################################### I just start

  2wTRQqoDM1A   https://www.youtube.com/watch?v=2wTRQqoDM1A
  Get Coursera Plus for 40% OFF 3 months: https://imp.i384100.net/c/3552395/3391150/14726 ▻ For more content lik

  -8qzzN7CWxA   https://www.youtube.com/watch?v=-8qzzN7CWxA
  This isn't a career most will succeed in.



Here's a clear structured answer on "How To Learn Machine Learning in 2025" based on the provided resources:

**Step-by-Step Guide to Learning Machine Learning**

1. **Prerequisites**: Start with essential mathematical and programming concepts, such as linear algebra, calculus, probability, and computer science fundamentals.
2. **Setup Environment**: Install necessary tools and libraries, including Python, scikit-learn, TensorFlow, or PyTorch.
3. **Key Concepts**: Learn about machine learning algorithms, data preprocessing, model evaluation, and interpretation.
4. **Algorithms**: Study popular algorithms like linear regression, decision trees, clustering, and neural networks.
5. **Real-World Projects**: Participate in machine learning competitions, build projects, or work on personal initiatives to apply concepts learned.
6. **Certification Programs**: Consider obtaining certifications from organizations like Google, IBM, or edX to demonstrate expertise.

**Recommended Resources**

* **GeeksforGeeks**: A comprehensive guide covering various aspects of machine learning from the basics to advanced topics.
* **IBM Machine Learning Guide**: Offers a structured approach to learning machine learning with modules and tutorials.
* **Google for Developers Machine Learning Crash Course**: Self-contained modules covering key concepts, algorithms, and projects.
* **DataCamp's Machine Learning in 2026**: A step-by-step guide to learning machine learning from scratch.

**Career Opportunities**

Machine learning is a rapidly growing field with various career paths, including:

* Data Scientist
* AI/ML Engineer
* Researcher
* Business Analyst

**Conclusion**

Learning machine learning requires dedication and persistence. By following this step-by-step guide, exploring recommended resources, and pursuing certification programs, individuals can develop the skills needed to succeed in this exciting field.


All data stored under search_id = 4


In [41]:
db_history()

| id | topic | type | web | yt | reports | created |
|----|-------|------|-----|----|---------|---------|
| 4 | i want to learn about Machine Learning | combined | 10 | 6 | 1 | 2026-03-15 00:35 |
| 3 | i want to learn about quantum computing  | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 2 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 1 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:31 |

[RealDictRow([('id', 4),
              ('topic', 'i want to learn about Machine Learning'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 35, 57, 161382, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 3),
              ('topic',
               'i want to learn about quantum computing breakthroughs in 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 44, 35126, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 2),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 20, 763229, tzinfo=datetime.timezone.u

----------------------------


## 9. LearnFlow — Personalised Learning Graph

A full graph-based adaptive learning system running entirely on Ollama + DuckDuckGo.

```
[START]
   │
   ▼
[generate_assessment]  ── 8-question diagnostic quiz
   │
   ▼
[wait_for_assessment]  ── INTERRUPT: collect user answers
   │
   ▼
[analyze_level]        ── score answers → level + gaps
   │
   ▼
[research_resources]   ── web search for relevant material
   │
   ▼
[build_roadmap]        ── personalised 4-6 module plan
   │
   ▼
[generate_content]  ◄──────────────────────────────────────┐
   │                                                        │ (failed)
   ▼                                                        │
[wait_for_quiz_req]    ── INTERRUPT: 'quiz' | 'more'        │
   │                                                        │
   ▼                                                        │
[generate_quiz]        ── 5 quiz questions from content     │
   │                                                        │
   ▼                                                        │
[wait_for_quiz_ans]    ── INTERRUPT: collect quiz answers   │
   │                                                        │
   ▼                                                        │
[evaluate_quiz]        ── score, feedback, save to DB       │
   │                                                        │
   ▼                                                        │
[quiz_router] ── score ≥ 70 → [advance] ── more → generate_content ──┘
                  score < 70 ────────────────────────────────────────┘
                                           done → [END] 🏆
```


In [32]:

import json
from dataclasses import dataclass, field
from typing import Optional

# ── LearnFlow State ───────────────────────────────────────────────────────────

@dataclass
class LearnFlowState:
    topic:                str   = ""
    assessment_questions: list  = field(default_factory=list)
    assessment_answers:   list  = field(default_factory=list)
    level:                str   = ""          # beginner | intermediate | advanced
    gaps:                 list  = field(default_factory=list)
    resources:            list  = field(default_factory=list)
    roadmap:              list  = field(default_factory=list)
    current_module:       int   = 0
    content:              str   = ""
    quiz:                 list  = field(default_factory=list)
    quiz_answers:         list  = field(default_factory=list)
    quiz_score:           float = 0.0
    attempts:             int   = 0
    completed:            bool  = False
    search_id:            Optional[int] = None


# ── Minimal graph engine (LangGraph-style) ────────────────────────────────────

END = "__END__"


class LearnFlowGraph:
    def __init__(self):
        self.nodes              = {}
        self.edges              = {}
        self.conditional_edges  = {}
        self._entry             = None

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def set_entry_point(self, name):
        self._entry = name

    def add_edge(self, src, dst):
        self.edges[src] = dst

    def add_conditional_edges(self, src, condition_fn, mapping):
        self.conditional_edges[src] = (condition_fn, mapping)

    def run(self, state):
        current = self._entry
        while current and current != END:
            print(f"\n{'─'*50}\n[GRAPH] ▶  {current}\n{'─'*50}")
            state = self.nodes[current](state)
            if current in self.conditional_edges:
                cond_fn, mapping = self.conditional_edges[current]
                current = mapping.get(cond_fn(state), END)
            else:
                current = self.edges.get(current)
        print("\n🏆  LearnFlow complete!")
        return state


print("LearnFlowState & LearnFlowGraph defined.")


LearnFlowState & LearnFlowGraph defined.


In [33]:

# ── Nodes: assessment, level analysis, research, roadmap ──────────────────────

def generate_assessment(state):
    """Generate 8-question diagnostic quiz."""
    print(f"[1/4] Generating diagnostic assessment for: {state.topic}")
    prompt = f"""Create a diagnostic quiz of exactly 8 questions to assess knowledge of: "{state.topic}"

Rules:
- Mix difficulty: 2 easy, 4 medium, 2 hard
- Include multiple-choice (A/B/C/D) and short-answer questions
- Each question targets a distinct concept or skill

Return a JSON array — nothing else:
[{{"q":"question text", "type":"mcq"|"short",
   "options":["A)...","B)...","C)...","D)..."]|null,
   "answer":"correct answer", "concept":"concept tested"}}]"""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        questions = json.loads(match.group()) if match else []
    except Exception:
        questions = []
    if not questions:
        questions = [
            {"q": f"What is {state.topic}?",          "type": "short", "options": None, "answer": "", "concept": "definition"},
            {"q": f"Name a key application of {state.topic}.", "type": "short", "options": None, "answer": "", "concept": "application"},
        ]
    state.assessment_questions = questions[:8]
    print(f"  Generated {len(state.assessment_questions)} questions.")
    return state


def wait_for_assessment(state):
    """INTERRUPT: Display questions and collect user answers."""
    print(f"\n{'='*60}\n📋  DIAGNOSTIC ASSESSMENT — {state.topic}\n{'='*60}\n")
    answers = []
    for i, q in enumerate(state.assessment_questions, 1):
        print(f"Q{i}: {q['q']}")
        if q.get("options"):
            for opt in q["options"]:
                print(f"   {opt}")
            ans = input("   Your answer (A/B/C/D): ").strip().upper()
        else:
            ans = input("   Your answer: ").strip()
        answers.append(ans)
        print()
    state.assessment_answers = answers
    print("✅  Assessment submitted.")
    return state


def analyze_level(state):
    """Score answers, detect level and knowledge gaps."""
    print("[2/4] Analysing your level...")
    qa_pairs = "\n".join(
        f"Q{i+1}: {q['q']}\nGiven: {a}\nExpected: {q.get('answer','N/A')}\nConcept: {q.get('concept','')}"
        for i, (q, a) in enumerate(zip(state.assessment_questions, state.assessment_answers))
    )
    prompt = f"""Evaluate quiz answers for topic "{state.topic}":
{qa_pairs}

Return JSON — nothing else:
{{"level":"beginner|intermediate|advanced",
  "gaps":["gap1","gap2","gap3"],
  "score":0-100}}"""

    raw   = llm(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        result      = json.loads(match.group()) if match else {}
        state.level = result.get("level", "beginner")
        state.gaps  = result.get("gaps", [f"fundamentals of {state.topic}"])
    except Exception:
        state.level = "beginner"
        state.gaps  = [f"fundamentals of {state.topic}"]

    print(f"  Level : {state.level.upper()}")
    print(f"  Gaps  : {state.gaps}")
    return state


def research_resources(state):
    """Web search for learning resources tailored to level and gaps."""
    print("[3/4] Researching learning resources...")
    queries = [f"{state.topic} {state.level} tutorial"] + \
              [f"{g} explained" for g in state.gaps[:2]]

    seen, unique = set(), []
    for q in queries:
        for r in search_web(q, max_results=4):
            if r["url"] and r["url"] not in seen:
                seen.add(r["url"])
                unique.append(r)
        time.sleep(0.3)

    state.resources = unique[:12]
    print(f"  Found {len(state.resources)} resources.")
    return state


def build_roadmap(state):
    """Generate a personalised module plan."""
    print("[4/4] Building personalised roadmap...")
    resource_titles = "\n".join(f"- {r['title']}: {r['snippet'][:100]}" for r in state.resources[:6])
    gaps_str        = "\n".join(f"- {g}" for g in state.gaps)

    prompt = f"""Create a personalised learning roadmap:
Topic : {state.topic}
Level : {state.level}
Gaps  :
{gaps_str}

Available resources:
{resource_titles}

Design 4-6 progressive modules. Return a JSON array — nothing else:
[{{"module":1, "title":"...", "objective":"...",
   "concepts":["c1","c2"], "duration":"30 min"}}]"""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        roadmap = json.loads(match.group()) if match else []
    except Exception:
        roadmap = []
    if not roadmap:
        roadmap = [
            {"module": 1, "title": f"Intro to {state.topic}",     "objective": "Understand basics",   "concepts": ["overview"],  "duration": "30 min"},
            {"module": 2, "title": f"Core {state.topic} concepts", "objective": "Build core knowledge","concepts": ["core"],      "duration": "45 min"},
            {"module": 3, "title": f"Applied {state.topic}",       "objective": "Practical skills",    "concepts": ["practice"],  "duration": "60 min"},
        ]

    state.roadmap        = roadmap
    state.current_module = 0

    print("\n📚  Your Learning Roadmap:")
    for m in roadmap:
        print(f"   Module {m.get('module','?')}: {m.get('title','?')}  ({m.get('duration','?')})")
    return state


print("Assessment, analysis, research & roadmap nodes defined.")


Assessment, analysis, research & roadmap nodes defined.


In [34]:

# ── Nodes: content, quiz, evaluation, routing ─────────────────────────────────

def generate_content(state):
    """Generate lesson content for the current module."""
    module = state.roadmap[state.current_module]
    total  = len(state.roadmap)
    print(f"Generating content — Module {state.current_module + 1}/{total}: {module.get('title')}")

    context = "\n\n".join(
        f"{r['title']}: {r['snippet']}"
        for r in state.resources[:5] if r.get("snippet")
    )
    system = "You are an expert tutor. Write clear, engaging lesson content with concrete examples."
    prompt = f"""Write a comprehensive lesson for:
Module: {module.get('title')}
Objective: {module.get('objective')}
Concepts: {', '.join(module.get('concepts', []))}
Learner level: {state.level}

Reference material:
{context[:2000]}

Structure with: ## Overview, ## Key Concepts (with examples), ## Practical Application, ## Summary
Use markdown formatting."""

    state.content  = llm(prompt, system=system)
    state.attempts = 0  # reset per module

    print(f"\n{'='*60}\n📖  MODULE {state.current_module + 1}: {module.get('title')}\n{'='*60}")
    display(Markdown(state.content))
    return state


def wait_for_quiz_req(state):
    """INTERRUPT: Wait for user to request the quiz or re-read the lesson."""
    print("\n" + "─"*60)
    print("  Type  'quiz'  to test your knowledge.")
    print("  Type  'more'  to re-read the content.")
    print("─"*60)
    while True:
        cmd = input("  Your choice: ").strip().lower()
        if cmd == "quiz":
            break
        elif cmd == "more":
            display(Markdown(state.content))
        else:
            print("  Please type 'quiz' or 'more'.")
    return state


def generate_quiz(state):
    """Generate 5 quiz questions from lesson content."""
    module = state.roadmap[state.current_module]
    print("Generating quiz questions...")
    prompt = f"""Create exactly 5 quiz questions to test understanding of:
Module: {module.get('title')}
Content: {state.content[:1500]}

Mix: 3 multiple-choice (A/B/C/D) + 2 short-answer.
Return a JSON array:
[{{"q":"...", "type":"mcq"|"short", "options":["A)...","B)...","C)...","D)..."]|null, "answer":"correct answer"}}]
Return ONLY valid JSON."""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        state.quiz = json.loads(match.group())[:5] if match else []
    except Exception:
        state.quiz = []
    if not state.quiz:
        state.quiz = [{"q": f"Summarise what you learned about {module.get('title')}.",
                       "type": "short", "options": None, "answer": ""}]
    return state


def wait_for_quiz_ans(state):
    """INTERRUPT: Display quiz and collect user answers."""
    module = state.roadmap[state.current_module]
    print(f"\n{'='*60}\n🧠  QUIZ — Module {state.current_module + 1}: {module.get('title')}\n{'='*60}\n")
    answers = []
    for i, q in enumerate(state.quiz, 1):
        print(f"Q{i}: {q['q']}")
        if q.get("options"):
            for opt in q["options"]:
                print(f"   {opt}")
            ans = input("   Your answer (A/B/C/D): ").strip().upper()
        else:
            ans = input("   Your answer: ").strip()
        answers.append(ans)
        print()
    state.quiz_answers = answers
    return state


def evaluate_quiz(state):
    """Score quiz and update knowledge tracking."""
    module = state.roadmap[state.current_module]
    print("Evaluating quiz...")
    qa_pairs = "\n".join(
        f"Q{i+1}: {q['q']}\nGiven: {a}\nCorrect: {q.get('answer','')}"
        for i, (q, a) in enumerate(zip(state.quiz, state.quiz_answers))
    )
    prompt = f"""Grade this quiz for module "{module.get('title')}":
{qa_pairs}

Partial credit for short-answer if the concept is correct.
Return JSON: {{"score": 0-100, "passed": true|false,
  "feedback": [{{"q":1, "correct":true|false, "explanation":"..."}}]}}
Passing threshold: 70. Return ONLY valid JSON."""

    raw   = llm(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        result         = json.loads(match.group()) if match else {}
        state.quiz_score = float(result.get("score", 0))
        feedback         = result.get("feedback", [])
    except Exception:
        state.quiz_score = 50.0
        feedback         = []

    state.attempts += 1
    print(f"\n📊  Score: {state.quiz_score:.0f}/100  (attempt {state.attempts})")
    for fb in feedback:
        icon = "✅" if fb.get("correct") else "❌"
        print(f"   {icon} Q{fb.get('q','?')}: {fb.get('explanation','')[:90]}")

    try:
        _lf_save_result(state)
    except Exception:
        pass
    return state


# ── Routing functions ─────────────────────────────────────────────────────────

def quiz_router(state):
    if state.quiz_score >= 70:
        print("  ✅  Passed!")
        return "passed"
    elif state.attempts >= 3:
        print("  ⚠️   Max attempts reached — advancing anyway.")
        return "passed"
    else:
        print(f"  ❌  Score {state.quiz_score:.0f} < 70. Reviewing content again…")
        return "failed"


def advance(state):
    state.current_module += 1
    print(f"  ➡️   Module {state.current_module + 1} / {len(state.roadmap)}")
    return state


def completion_router(state):
    if state.current_module >= len(state.roadmap):
        state.completed = True
        print("\n🏆  All modules completed!")
        return "done"
    return "more"


print("Content, quiz & routing nodes defined.")


Content, quiz & routing nodes defined.


In [35]:

# ── PostgreSQL schema for LearnFlow ──────────────────────────────────────────
def init_learnflow_db():
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS learnflow_sessions (
                    id         SERIAL PRIMARY KEY,
                    topic      TEXT NOT NULL,
                    level      TEXT,
                    gaps       JSONB,
                    roadmap    JSONB,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS learnflow_results (
                    id           SERIAL PRIMARY KEY,
                    session_id   INT REFERENCES learnflow_sessions(id) ON DELETE CASCADE,
                    module_idx   INT,
                    module_title TEXT,
                    score        FLOAT,
                    attempts     INT,
                    passed       BOOLEAN,
                    created_at   TIMESTAMPTZ DEFAULT NOW()
                );
            """)
        conn.commit()
    print("LearnFlow DB tables ready.")


def _lf_save_result(state):
    module = state.roadmap[state.current_module] if state.current_module < len(state.roadmap) else {}
    if state.search_id is None:
        with get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "INSERT INTO learnflow_sessions (topic, level, gaps, roadmap)"
                    " VALUES (%s,%s,%s,%s) RETURNING id",
                    (state.topic, state.level,
                     json.dumps(state.gaps), json.dumps(state.roadmap))
                )
                state.search_id = cur.fetchone()[0]
            conn.commit()

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO learnflow_results"
                " (session_id, module_idx, module_title, score, attempts, passed)"
                " VALUES (%s,%s,%s,%s,%s,%s)",
                (state.search_id, state.current_module,
                 module.get("title", ""), state.quiz_score,
                 state.attempts, state.quiz_score >= 70)
            )
        conn.commit()


try:
    init_learnflow_db()
except Exception as e:
    print(f"[DB] Not available — results won't be persisted: {e}")


# ── Graph assembly ────────────────────────────────────────────────────────────
def build_learnflow_graph():
    g = LearnFlowGraph()

    g.add_node("generate_assessment", generate_assessment)
    g.add_node("wait_for_assessment", wait_for_assessment)
    g.add_node("analyze_level",       analyze_level)
    g.add_node("research_resources",  research_resources)
    g.add_node("build_roadmap",       build_roadmap)
    g.add_node("generate_content",    generate_content)
    g.add_node("wait_for_quiz_req",   wait_for_quiz_req)
    g.add_node("generate_quiz",       generate_quiz)
    g.add_node("wait_for_quiz_ans",   wait_for_quiz_ans)
    g.add_node("evaluate_quiz",       evaluate_quiz)
    g.add_node("advance",             advance)

    g.set_entry_point("generate_assessment")

    # Linear edges
    g.add_edge("generate_assessment", "wait_for_assessment")
    g.add_edge("wait_for_assessment", "analyze_level")
    g.add_edge("analyze_level",       "research_resources")
    g.add_edge("research_resources",  "build_roadmap")
    g.add_edge("build_roadmap",       "generate_content")
    g.add_edge("generate_content",    "wait_for_quiz_req")
    g.add_edge("wait_for_quiz_req",   "generate_quiz")
    g.add_edge("generate_quiz",       "wait_for_quiz_ans")
    g.add_edge("wait_for_quiz_ans",   "evaluate_quiz")

    # Conditional: quiz_router — passed → advance, failed → retry content
    g.add_conditional_edges("evaluate_quiz", quiz_router, {
        "passed": "advance",
        "failed": "generate_content",
    })

    # Conditional: completion_router — more → next module, done → END
    g.add_conditional_edges("advance", completion_router, {
        "more": "generate_content",
        "done": END,
    })

    return g


print("LearnFlow graph assembled. Run build_learnflow_graph() to start.")


LearnFlow DB tables ready.
LearnFlow graph assembled. Run build_learnflow_graph() to start.


In [ ]:
# ── RUN LEARNFLOW ────────────────────────────────────────────────────────────
LF_TOPIC = "i want to ML"   
# ─────────────────────────────────────────────────────────────────────────────

graph = build_learnflow_graph()
initial_state = LearnFlowState(topic=LF_TOPIC)
final_state = graph.run(initial_state)



──────────────────────────────────────────────────
[GRAPH] ▶  generate_assessment
──────────────────────────────────────────────────
[1/4] Generating diagnostic assessment for: i want to ML
  Generated 2 questions.

──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_assessment
──────────────────────────────────────────────────

📋  DIAGNOSTIC ASSESSMENT — i want to ML

Q1: What is i want to ML?

Q2: Name a key application of i want to ML.

✅  Assessment submitted.

──────────────────────────────────────────────────
[GRAPH] ▶  analyze_level
──────────────────────────────────────────────────
[2/4] Analysing your level...
  Level : BEGINNER
  Gaps  : ['fundamentals of i want to ML']

──────────────────────────────────────────────────
[GRAPH] ▶  research_resources
──────────────────────────────────────────────────
[3/4] Researching learning resources...
  Found 8 resources.

──────────────────────────────────────────────────
[GRAPH] ▶  build_roadmap
────────────────────

# Introduction to ML: A Beginner's Guide

## Module Overview

This module is designed for beginners interested in machine learning (ML). We will explore the basics of ML, covering key concepts, algorithms, and practical applications.

## Learning Objectives

By the end of this module, you will be able to:

* Understand the basics of machine learning
* Identify the response variable (target) and output (predictable)
* Recognize the importance of data preprocessing and model evaluation
* Apply ML concepts to a real-world scenario

## Concepts Overview

### 1. Response Variable (Target)

The response variable, also known as the target or dependent variable, is the output you want to predict. It's the goal of the prediction task.

Example: Order Canceled - Is it Yes or No?

### 2. Output (Predictable)

The predictable variable is what we want to predict. In this example, it's whether an order will be canceled (Yes or No).

### 3. Data

Data is used to train and test machine learning models. It's essential to preprocess the data before building a model.

## Practical Application: Customer Churn Prediction

Let's use customer churn prediction as a practical application.

Suppose we have a dataset with the following columns:

| Column Name | Description |
| --- | --- |
| Age | Age of the customer |
| Income | Monthly income of the customer |
| Subscriptions | Number of subscriptions (active or inactive) |
| Churned | Whether the customer has churned or not |

We want to build a model that predicts whether a customer will churn based on their age, income, and number of active subscriptions.

## Summary

Machine learning is a powerful tool for analyzing data and making predictions. In this module, we covered the basics of ML, including response variables, outputs, and data preprocessing. We also applied ML concepts to a practical scenario, demonstrating its potential in real-world problems like customer churn prediction.

### Key Concepts

* Response Variable (Target)
* Output (Predictable)
* Data
* Machine Learning Algorithm:
	+ Classification
	+ Regression

### Practical Exercises

1. Implement a simple classification model using Python and scikit-learn to predict whether an order will be canceled or not.
2. Preprocess your own customer data and apply it to a machine learning model to predict churn.

### References

* [ML.NET Tutorial - Get started in 10 minutes](https://docs.microsoft.com/en-us/azure/machine-learning/how-to-get-started-machine-learning)
* Machine Learning 101: The Complete Beginner’s Guide to Machine Learning by Amit Verma (Medium, August 2025)
* Machine Learning for Beginners: Introduction to Machine Learning for Beginners (4 days ago, Medium)

### Next Steps

In the next module, we'll dive deeper into machine learning algorithms and practical applications. We'll also explore data preprocessing techniques and model evaluation metrics.

Let me know if you have any questions or need further clarification on any of the concepts covered in this module!


──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_req
──────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Type  'quiz'  to test your knowledge.
  Type  'more'  to re-read the content.
────────────────────────────────────────────────────────────

──────────────────────────────────────────────────
[GRAPH] ▶  generate_quiz
──────────────────────────────────────────────────
Generating quiz questions...

──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_ans
──────────────────────────────────────────────────

🧠  QUIZ — Module 1: Intro to i want to ML

Q1: Summarise what you learned about Intro to i want to ML.


──────────────────────────────────────────────────
[GRAPH] ▶  evaluate_quiz
──────────────────────────────────────────────────
Evaluating quiz...

📊  Score: 100/100  (attempt 1)
   ✅ Q1: Machine Learning is the science of building hardware or software that can achieve t

**Module: Core**
======================================================

### Objective:
By the end of this module, you will have a solid foundation in core ML concepts and be able to apply them to real-world scenarios.

### Concepts: Core
----------------

#### 1. Response Variable (Target Variable)
------------------------------------------

A response variable, also known as a target or dependent variable, is the output that we want to predict. In the context of machine learning, it's the result of our prediction task. For example, in the **Order Canceled** classification problem, the response variable is "Yes" if the order was canceled and "No" otherwise.

#### 2. Output (Target Variable)
------------------------------

The output or target variable is a vector of values that we want to predict for each sample in our dataset. In the Order Canceled example, the output variable has two possible values: "Yes" and "No".

#### 3. Predictive Model
----------------------

A predictive model is a mathematical function that takes input data and produces output predictions. In machine learning, it's an algorithm that makes predictions on new, unseen data.

### Learner Level:
----------------

This module is designed for beginners who want to build core knowledge in machine learning concepts. We'll cover the basics of response variables, outputs, predictive models, and more.

**Lesson 1: Introduction to Core ML Concepts**
-----------------------------------------------

#### Step 1: Understanding Response Variables
-----------------------------------------

A response variable is a key concept in machine learning that represents the output or target variable for our model. In this example, we have a simple classification problem where we want to predict whether an order was canceled (Yes/No) based on some input features.

```python
import pandas as pd

# Create a sample dataset
data = {
    'Order ID': [1, 2, 3, 4, 5],
    'Customer Name': ['John', 'Jane', 'Bob', 'Alice', 'Mike'],
    'Total Amount': [100.0, 200.0, 300.0, 400.0, 500.0]
}
df = pd.DataFrame(data)

# Define the response variable (Target Variable)
target_variable = df['Order Canceled']
```

#### Step 2: Understanding Output
------------------------------

The output or target variable is a vector of values that we want to predict for each sample in our dataset. In this example, we have two possible values: "Yes" and "No".

```python
# Define the output (Target Variable)
output_variable = ['Yes', 'No']
```

#### Step 3: Understanding Predictive Models
-----------------------------------------

A predictive model is a mathematical function that takes input data and produces output predictions. In machine learning, it's an algorithm that makes predictions on new, unseen data.

```python
# Define a simple predictive model using linear regression (for demonstration purposes only)
def predict_order_canceled(df):
    # Extract the input features from the dataset
    features = df[['Customer Name', 'Total Amount']]
    
    # Calculate the output variable (target variable) using linear regression
    predicted_values = [0.5 * x + 10 for x in features]
    
    return predicted_values

# Use the predictive model to make predictions on new data
new_data = pd.DataFrame({
    'Customer Name': ['Emily'],
    'Total Amount': [500.0]
})
predicted_order_canceled = predict_order_canceled(new_data)
print(predicted_order_canceled)  # Output: [250.0]
```

**Summary**
----------

In this lesson, we covered the basics of response variables, outputs, and predictive models in machine learning concepts. We also demonstrated a simple predictive model using linear regression.

### Next Steps
----------------

* Practice building predictive models using different algorithms (e.g., decision trees, random forests)
* Explore real-world datasets and use them to build and evaluate your own predictive models
* Read more about core ML concepts and techniques in our next lesson.


──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_req
──────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Type  'quiz'  to test your knowledge.
  Type  'more'  to re-read the content.
────────────────────────────────────────────────────────────


**Module: Core**
======================================================

### Objective:
By the end of this module, you will have a solid foundation in core ML concepts and be able to apply them to real-world scenarios.

### Concepts: Core
----------------

#### 1. Response Variable (Target Variable)
------------------------------------------

A response variable, also known as a target or dependent variable, is the output that we want to predict. In the context of machine learning, it's the result of our prediction task. For example, in the **Order Canceled** classification problem, the response variable is "Yes" if the order was canceled and "No" otherwise.

#### 2. Output (Target Variable)
------------------------------

The output or target variable is a vector of values that we want to predict for each sample in our dataset. In the Order Canceled example, the output variable has two possible values: "Yes" and "No".

#### 3. Predictive Model
----------------------

A predictive model is a mathematical function that takes input data and produces output predictions. In machine learning, it's an algorithm that makes predictions on new, unseen data.

### Learner Level:
----------------

This module is designed for beginners who want to build core knowledge in machine learning concepts. We'll cover the basics of response variables, outputs, predictive models, and more.

**Lesson 1: Introduction to Core ML Concepts**
-----------------------------------------------

#### Step 1: Understanding Response Variables
-----------------------------------------

A response variable is a key concept in machine learning that represents the output or target variable for our model. In this example, we have a simple classification problem where we want to predict whether an order was canceled (Yes/No) based on some input features.

```python
import pandas as pd

# Create a sample dataset
data = {
    'Order ID': [1, 2, 3, 4, 5],
    'Customer Name': ['John', 'Jane', 'Bob', 'Alice', 'Mike'],
    'Total Amount': [100.0, 200.0, 300.0, 400.0, 500.0]
}
df = pd.DataFrame(data)

# Define the response variable (Target Variable)
target_variable = df['Order Canceled']
```

#### Step 2: Understanding Output
------------------------------

The output or target variable is a vector of values that we want to predict for each sample in our dataset. In this example, we have two possible values: "Yes" and "No".

```python
# Define the output (Target Variable)
output_variable = ['Yes', 'No']
```

#### Step 3: Understanding Predictive Models
-----------------------------------------

A predictive model is a mathematical function that takes input data and produces output predictions. In machine learning, it's an algorithm that makes predictions on new, unseen data.

```python
# Define a simple predictive model using linear regression (for demonstration purposes only)
def predict_order_canceled(df):
    # Extract the input features from the dataset
    features = df[['Customer Name', 'Total Amount']]
    
    # Calculate the output variable (target variable) using linear regression
    predicted_values = [0.5 * x + 10 for x in features]
    
    return predicted_values

# Use the predictive model to make predictions on new data
new_data = pd.DataFrame({
    'Customer Name': ['Emily'],
    'Total Amount': [500.0]
})
predicted_order_canceled = predict_order_canceled(new_data)
print(predicted_order_canceled)  # Output: [250.0]
```

**Summary**
----------

In this lesson, we covered the basics of response variables, outputs, and predictive models in machine learning concepts. We also demonstrated a simple predictive model using linear regression.

### Next Steps
----------------

* Practice building predictive models using different algorithms (e.g., decision trees, random forests)
* Explore real-world datasets and use them to build and evaluate your own predictive models
* Read more about core ML concepts and techniques in our next lesson.


──────────────────────────────────────────────────
[GRAPH] ▶  generate_quiz
──────────────────────────────────────────────────
Generating quiz questions...

──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_ans
──────────────────────────────────────────────────

🧠  QUIZ — Module 2: Core i want to ML concepts

Q1: Summarise what you learned about Core i want to ML concepts.


──────────────────────────────────────────────────
[GRAPH] ▶  evaluate_quiz
──────────────────────────────────────────────────
Evaluating quiz...

📊  Score: 60/100  (attempt 1)
   ✅ Q1: The quiz did not provide a summary of Core i want to ML concepts.
  ❌  Score 60 < 70. Reviewing content again…

──────────────────────────────────────────────────
[GRAPH] ▶  generate_content
──────────────────────────────────────────────────
Generating content — Module 2/3: Core i want to ML concepts

📖  MODULE 2: Core i want to ML concepts


**Module: Core - Building Core Knowledge**

**Objective:** Build core knowledge in machine learning concepts for beginners.

**Concepts:**

### 1. What is Machine Learning?

Machine learning is a subset of artificial intelligence that involves training algorithms on data to make predictions or decisions without being explicitly programmed.

* **Response Variable (a.k.a. Target, Dependent Variable):**
 + A categorical variable representing the outcome or target value.
 + Example: Order Canceled (Yes or No).
 + Data should be collected for this response variable.

### 2. Machine Learning Algorithms

Machine learning algorithms are used to classify data into specific categories based on certain rules or patterns.

* **Classification:** The process of predicting a categorical variable based on feature values.
* **Key Algorithm:**
 + Logistic Regression
 + Decision Trees
 + Random Forests
 + Support Vector Machines (SVMs)

### 3. Data Preprocessing

Data preprocessing is the process of cleaning, transforming, and formatting data to prepare it for machine learning models.

* **Importing Libraries:**
 + `System.Numerics`
 + `System.Linq`
 + `Scikit-learn`

**Learner Level:** Beginner

**Code Example:**

```csharp
using System;
using System.Data;
using System.Linq;
using Microsoft.ML;

class Program
{
    static void Main()
    {
        // Importing Libraries
        var mlContext = new MLContext();
        var dataView = mlContext.Data.LoadFromTextFile("orders.csv", hasHeader: true);
        
        // Data Preprocessing
        var dataPreprocessor = new DataPreprocessor();
        var transformedData = dataPreprocessor.Transform(dataView);
        
        // Model Training
        var model = mlContext.Model.CreateModel(mlContext.Transforms.Text.FeaturizeText(transformedData, {"feature1", "feature2"}));
    }
}
```

### 4. Model Evaluation

Model evaluation is the process of assessing the performance of a machine learning model on a test dataset.

* **Metrics:**
 + Accuracy
 + Precision
 + Recall
 + F1 Score

**Code Example:**

```csharp
using System;
using Microsoft.ML;

class Program
{
    static void Main()
    {
        // Importing Libraries
        var mlContext = new MLContext();
        
        // Model Evaluation
        var model = mlContext.Model.Load("order_canceled_model");
        var testDataView = mlContext.Data.LoadFromTextFile("test_data.csv", hasHeader: true);
        var predictions = model.Predict(testDataView);
        
        // Evaluation Metrics
        var accuracy = CalculateAccuracy(predictions, testDataView);
        Console.WriteLine($"Accuracy: {accuracy}");
    }
}

// Helper Method to calculate Accuracy
static double CalculateAccuracy(double[] predictions, DataTable testDataTable)
{
    int correctPredictions = 0;
    
    foreach (DataRow row in testDataTable.Rows)
    {
        if ((double)row["output"] == "Yes") // Replace with actual output values
        {
            if (((bool)row["prediction"]) == (bool)(predictions[0] == "Yes"))
            {
                correctPredictions++;
            }
        }
    }
    
    return (double)correctPredictions / testDataTable.Count;
}
```

### 5. Model Deployment

Model deployment is the process of deploying a trained machine learning model to production.

* **Environment:** Azure, Local Machine
* **Deployment Method:**
 + Azure: Use Azure Pipelines for Continuous Integration and Deployment.
 + Local Machine: Use the `mlContext.Model` API to deploy the model locally.

**Code Example:**

```csharp
using System;
using Microsoft.ML;

class Program
{
    static void Main()
    {
        // Importing Libraries
        var mlContext = new MLContext();
        
        // Model Deployment
        mlContext.Model.Save("order_canceled_model", mlContext.Model);
    }
}
```

By following this comprehensive lesson, beginners will gain a solid understanding of machine learning concepts, including response variables, classification algorithms, data preprocessing, model evaluation, and deployment. They will be able to apply these concepts to real-world scenarios and deploy their own machine learning models using the ML.NET CLI and Azure Pipelines.


──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_req
──────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Type  'quiz'  to test your knowledge.
  Type  'more'  to re-read the content.
────────────────────────────────────────────────────────────
